# 📘 Tutorial 7: Grouping, Aggregation, and Replicates

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

Many datasets contain repeated measurements. Before plotting means or error bars, we need to group observations and compute summaries without losing the raw data.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- use `groupby`,
- calculate mean, standard deviation, and count,
- aggregate replicate measurements,
- calculate standard error,
- produce summary tables for later error bars,
- explain why summaries should not permanently replace raw data.


## 🔧 Setup


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Read replicate measurements


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

replicate_path = project_root / "data" / "part1" / "replicate_measurements.csv"
raw_replicates = pd.read_csv(replicate_path)
raw_replicates


Each row is one replicate measurement. The raw table is the evidence. Summary tables are derived from it.


## 2. A first `groupby`


In [ ]:
raw_replicates.groupby("condition")["response"].mean()


This gives the mean response for each condition, but a plotting table usually needs more than a mean.


## 3. Aggregate mean, standard deviation, and count


In [ ]:
summary = (
    raw_replicates.groupby(["condition"], as_index=False)
    .agg(
        mean_response=("response", "mean"),
        sd_response=("response", "std"),
        n=("response", "count"),
    )
)

summary


Named aggregation creates columns with clear names. That makes the result easier to use later.


## 4. Standard error


In [ ]:
summary["sem_response"] = summary["sd_response"] / (summary["n"] ** 0.5)
summary


The standard error of the mean is often used for uncertainty around a mean:

$$\mathrm{SEM} = \frac{s}{\sqrt{n}}$$

Whether SEM is appropriate depends on the scientific question. The important point here is to calculate it explicitly.


## 5. Preserve raw and summary tables


In [ ]:
print("Raw rows:", len(raw_replicates))
print("Summary rows:", len(summary))


In [ ]:
raw_replicates.head()


In [ ]:
summary


The summary table is useful for error bars. The raw table is still needed for checking outliers, auditing replicates, and choosing other summaries later.


## 6. Group by multiple columns


In [ ]:
by_sample = (
    raw_replicates.groupby(["condition", "sample_id"], as_index=False)
    .agg(
        mean_response=("response", "mean"),
        sd_response=("response", "std"),
        n=("response", "count"),
    )
)

by_sample


Grouping by multiple columns is common when a plot has both a main variable and a grouping variable.


## 7. Build an error-bar table


In [ ]:
error_bar_table = summary[
    ["condition", "mean_response", "sd_response", "sem_response", "n"]
].copy()

error_bar_table


A later Matplotlib tutorial could use `condition` for x positions, `mean_response` for bar heights or points, and `sem_response` for error bars.


## 8. Checkpoint: summary by sample

Task: calculate the mean response and replicate count for each `sample_id`.


In [ ]:
sample_summary = (
    raw_replicates.groupby("sample_id", as_index=False)
    .agg(mean_response=("response", "mean"), n=("response", "count"))
)

sample_summary


## 🧭 Key takeaways

- `groupby` splits a table into groups before summarizing.
- Use named aggregation for readable summary columns.
- Mean, standard deviation, count, and SEM are common pre-plotting summaries.
- Summary tables are derived products, not replacements for raw data.
- Error-bar tables should state exactly what uncertainty column they contain.
